# A07: Proyecto Integrador Final - Motor de Procesamiento Asíncrono de Datos

Nivel: **Avanzado** | Proyecto integrador final | Integra todo el curso


## 1. Objetivos del Proyecto

Este es el **proyecto integrador final** del nivel Avanzado. Sintetiza TODAS las habilidades del curso en una sola pieza de software realista: un **Motor de Procesamiento Asíncrono de Datos** con pipeline concurrente y resiliente.

Al finalizar, habrás demostrado la capacidad de:

1. **Diseñar** un pipeline de datos concurrente (productor → cola → trabajadores → sink) con arquitectura profesional.
2. **aplicar** `asyncio` avanzado: `Semaphore` (límite de concurrencia), `Queue`, `gather`, `asyncio.run`, `asyncio.timeout`.
3. **Modelar** el dominio con `dataclasses` (`Job`, `JobResult`, `Stats`) y tipado estricto con `typing` /`Protocol` / genéricos.
4. **Construir** POO avanzada: herencia, clases abstractas (`ABC`), `Protocol` (duck typing estático), composición y polimorfismo.
5. **Implementar** resiliencia: `retry` con backoff exponencial y una cola dedicada de fallos (Dead Letter Queue).
6. **Registrar** actividad con `logging` estructurado (formato, niveles, componentes por logger).
7. **Usar** metaprogramación y patrones: decorador `@retry`, factory de procesadores, patrón Strategy + Observer.
8. **Medir** métricas: contadores de completados/errores/reintentos, tiempos, reporte final y rate limiting.
9. **Probar** el sistema con `unittest.IsolatedAsyncioTestCase` de la stdlib (sin dependencias externas).

> **Solo se usa la librería estándar** para garantizar portabilidad y demostrar el poder de asyncio + typing sin dependencias.


## 2. Analogía: La Cocina de Alta Cocina

Imagina un **restaurante de alta cocina** con un servicio impecable:

- **Productor** = El camarero que toma los pedidos de los clientes y los coloca en la bandeja de la cocina.
- **Cola de entrada** = La bandeja donde esperan los pedidos pendientes de preparación.
- **Trabajadores** = Los chefs. Cada uno prepara un pedido (job) a la vez, y solo caben `N` chefs en la cocina (semáforo).
- **Agregador / Sink** = El encargado de salón que entrega los platos y lleva la cuenta de lo servido (resultados y métricas).
- **Dead Letter Queue** = Pedidos que se quemaron y se apartan aparte para revisión, sin bloquear al resto.
- **Retry + backoff** = Si un plato sale mal, el chef espera un poco más en cada intento y lo vuelve a intentar.
- **Rate limiting** = El restaurante solo acepta un máximo de pedidos por minuto para no saturar la cocina.

```ascii
        ┌──────────────────────────────────────────────────────────────┐
        │              MOTOR DE PROCESAMIENTO ASÍNCRONO                 │
        └──────────────────────────────────────────────────────────────┘

  ┌──────────────┐  jobs   ┌────────────────┐      ┌───────────────────┐
  │  PRODUCTOR    │ ───────►│ COLA de entrada │      │  TRABAJADORES     │
  │ (genera jobs) │  rate   │  (asyncio.Queue)│      │  (workers async)  │
  └──────────────┘  limit   └───────┬────────┘      │                   │
                                        │              │  Chef/Trabajador 1│────┐
                                        │              │  Chef/Trabajador 2│────┤
                           Semáforo   │              │  Chef/Trabajador 3│────┤
                           limita a N  │              │  ...             │    │
                           simultáneos  │              └────────┬─────────┘    │
                                        ▼                       │  procesa      │
                             ┌─────────────────────┐             │  con retry    │
                             │  asyncio.Semaphore  │             ▼               │
                             └─────────────────────┘      ┌───────────┐  resultado
                                        │                  │ PROCESADOR │────────►
                                        ▼                  └───────────┘  (estrategia)
                          ┌──────────────────────┐              ▲
                          │ COLA de fallos (DLQ) │◄── error ────┘
                          │ (jobs fallidos aparte)│
                          └──────────────────────┘
                                        │  resultados
                                        ▼
                             ┌─────────────────────┐
                             │  SINK / AGREGADOR   │
                             │  (stats + reporte)  │
                             └─────────────────────┘
```


## 3. Requisitos del Motor

| # | Requisito | Implementación |
|---|-----------|----------------|
| 1 | Motor asíncrono que procesa N jobs de forma concurrente | `asyncio` + `Trabajador` |
| 2 | Limitar concurrencia (máx. jobs simultáneos) | `asyncio.Semaphore` |
| 3 | Cada job: lee de cola → procesa → registra resultado | `Queue` + worker + sink |
| 4 | Pipeline rico: ingesta → procesamiento → salida | `Productor` → `Queue` → `Trabajador` → `Agregador` |
| 5 | Logging estructurado | `logging` con formato y loggers por componente |
| 6 | Manejo de errores resiliente (retry con backoff) | decorador `@retry` |
| 7 | Jobs fallidos a una cola dedicada (DLQ) | `Queue` secundaria |
| 8 | Métricas: completados / errores / tiempos / reintentos | `Stats` (dataclass) |
| 9 | Rate limiting (no exceder un ritmo de producción) | `asyncio.sleep` controlado |
| 10 | Extensibilidad (nuevos procesadores sin tocar el motor) | `Protocol` + factory |

### Contratos de diseño

- **`Job`**: descripción de una tarea de datos (id, tipo, payload, intentos máx., registro de intentos).
- **`JobResult`**: salida de procesar un job (job id, éxito, resultado, duración, errores).
- **`Stats`**: acumuladores del motor (completados, errores, reintentos, tiempos, contador por tipo).


## 4. Diseño y Arquitectura (Diagrama de Clases)

```ascii
            ┌──────────────────────────────────────────────────────────┐
            │                  MotorDeProcesamiento                     │
            │  orquestador: Semaphore, gather, colas, logging, stats    │
            └───────┬──────────────┬───────────────┬──────────┬────────┘
                    │              │               │          │
                    ▼              ▼               ▼          ▼
            ┌─────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐
            │  Productor  │ │ Trabajador │ │   Cola*    │ │  Agregador │
            │ (genera      │ │ (worker     │ │(entrada y   │ │ (Sink +    │
            │  jobs)      │ │  async)     │ │  DLQ)      │ │  stats)    │
            └─────────────┘ └─────┬───────┘ └────────────┘ └────────────┘
                                  │
                                  ▼
                           ┌──────────────┐
                           │  Procesador  │◄── Protocol/ABC (contrato)
                           └──────────────┘
                           ▲        ▲        ▲        ▲
                     ┌──────┘  ┌─────┘  ┌──────┘  ┌──────┘
               ValidarDatos  Normalizar  CalcularStats  Transformar
                           (factory los instancia por tipo)
```

### Tabla de componentes

| Clase / interfaz | Tipo | Responsabilidad | Concepto del curso |
|------------------|------|-----------------|--------------------|
| `Job` | `@dataclass` | Datos de una tarea de datos | dataclasses, `field` |
| `JobResult` | `@dataclass` | Salida de procesar un Job | dataclasses, defaults |
| `Stats` | `@dataclass` | Acumuladores y métricas | dataclasses + métodos |
| `Procesador` | `typing.Protocol` | Contrato de estrategia | Protocol / duck typing |
| `ProcesadorBase` | `ABC` | Base común (id, descripción) | abc/abstractmethod |
| `ColaDeTrabajos` | clase | Envoltorio de `asyncio.Queue` | genéricos `Queue[Job]` |
| `Productor` | clase | Genera jobs con rate limiting | asyncio.sleep, factory |
| `Trabajador` | clase | Consume cola, retry, métricas | decorador `@retry`, logging |
| `MotorDeProcesamiento` | clase | Orquestador (semáforo, gather) | Semaphore, gather, timeout |
| `Agregador` | clase | Sink: consolida resultados | composición / observer |

### Patrones aplicados

- **Strategy**: cada `Procesador` es una estrategia intercambiable (seleccionada por el tipo del job).
- **Factory**: `crear_procesador()` devuelve el procesador adecuado según el tipo (patrón Factory Method).
- **Observer**: el `Agregador` recibe notificaciones de `JobResult` y reacciona acumulando métricas.
- **Decorator**: `@retry` envuelve la función de procesamiento con reintentos y backoff.


## 5. Desarrollo Paso a Paso

### Paso 0: Imports + Configuración de Logging

Importamos toda la librería estándar que necesitaremos y configuramos un logging estructurado con formato, fecha y niveles. Además fijamos una **semilla** para resultados reproducibles.


In [ ]:
import asyncio
import logging
import random
import time
from abc import ABC, abstractmethod
from collections import deque
from dataclasses import dataclass, field
from typing import Any, Callable, Deque, Optional, Protocol

# Semilla para que los datos simulados sean reproducibles
random.seed(42)

# --- Configuración de logging estructurado ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)-22s | %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("motor").setLevel(logging.INFO)

print("Imports OK")


### Paso 1: Dataclasses del Dominio (`Job`, `JobResult`, `Stats`)

Definimos el modelo de datos con `dataclasses`. Usamos `frozen=True` para `Job` (inmutable, representación pura de entrada) y `field(default_factory=...)` para colecciones. El `Stats` encapsula la lógica de métricas.


In [ ]:
@dataclass(frozen=True)
class Job:
    """Tarea de datos a procesar por el motor."""
    id: int
    tipo: str                 # tipo de procesador a usar
    payload: Any              # datos reales a procesar
    intentos_max: int = 3     # máximo de intentos antes de fallar
    prioridad: int = 0        # mayor = más importante

    def __post_init__(self) -> None:
        if self.intentos_max < 1:
            raise ValueError("intentos_max debe ser >= 1")


@dataclass
class JobResult:
    """Resultado de procesar un Job (éxito o error)."""
    job_id: int
    tipo: str
    exito: bool
    resultado: Any = None
    duracion_s: float = 0.0
    num_intentos: int = 1
    errores: Deque[str] = field(default_factory=deque)


@dataclass
class Stats:
    """Métricas agregadas del motor."""
    completados: int = 0
    fallidos: int = 0
    reintentos: int = 0
    tiempo_total_s: float = 0.0
    por_tipo: dict[str, int] = field(default_factory=dict)
    tiempos_totales_s: float = 0.0

    def registrar(self, res: JobResult) -> None:
        """Actualiza las métricas con un resultado."""
        if res.exito:
            self.completados += 1
        else:
            self.fallidos += 1
        self.reintentos += max(0, res.num_intentos - 1)
        self.tiempos_totales_s += res.duracion_s
        self.por_tipo[res.tipo] = self.por_tipo.get(res.tipo, 0) + 1

    def registrar_tiempo_total(self, inicio: float) -> None:
        self.tiempo_total_s = time.monotonic() - inicio

    def resumen(self) -> str:
        return (
            f"Completados={self.completados} | Fallidos={self.fallidos} | "
            f"Reintentos={self.reintentos} | Tiempo=proceso {self.tiempos_totales_s:.2f}s "
            f"(real {self.tiempo_total_s:.2f}s) | Por tipo={dict(self.por_tipo)}"
        )


print("Dominio definido (Job, JobResult, Stats)")


### Paso 2: Procesadores (Patrón Strategy + Factory)

Definimos el contrato `Procesador` como `Protocol` (duck typing estático) y una `ProcesadorBase` abstracta. Cada procesador concreto implementa `procesar()`. Para simular trabajo real usamos `asyncio.sleep`, con una tasa de fallo configurable para probar el retry.


In [ ]:
class Procesador(Protocol):
    """Contrato que cualquier procesador debe cumplir."""
    id_procesador: str

    async def procesar(self, payload: Any) -> Any:
        """Procesa el payload y devuelve un resultado."""
        ...


class ProcesadorBase(ABC):
    """Base común: identidad + descripción + registro."""
    def __init__(self, tasa_fallo: float = 0.0) -> None:
        self.tasa_fallo = tasa_fallo
        self._log = logging.getLogger(f"motor.proc.{self.id_procesador}")

    @property
    @abstractmethod
    def id_procesador(self) -> str: ...

    @property
    def descripcion(self) -> str:
        return f"{self.id_procesador} (estado: {type(self).__name__})"

    def _simular_trabajo(self, base_ms: float = 50) -> float:
        """Simula tiempo de cómputo determinista + ruido."""
        return base_ms + random.uniform(0, base_ms)

    async def procesar(self, payload: Any) -> Any:
        raise NotImplementedError


class ValidarDatos(ProcesadorBase):
    """Valida que el payload tenga las claves esperadas."""
    @property
    def id_procesador(self) -> str:
        return "validar"

    async def procesar(self, payload: Any) -> Any:
        await asyncio.sleep(self._simular_trabajo() / 1000)
        if random.random() < self.tasa_fallo:
            raise ValueError("dato inválido (fallo simulado)")
        if not isinstance(payload, dict):
            raise TypeError("el payload debe ser un dict para validar")
        faltan = {"id", "monto"} - set(payload)
        if faltan:
            raise KeyError(f"faltan claves: {sorted(faltan)}")
        return {"validado": True, "id": payload["id"]}


class NormalizarTexto(ProcesadorBase):
    """Normaliza y limpia cadenas de texto."""
    @property
    def id_procesador(self) -> str:
        return "normalizar"

    async def procesar(self, payload: Any) -> Any:
        await asyncio.sleep(self._simular_trabajo(30) / 1000)
        if random.random() < self.tasa_fallo:
            raise ValueError("texto corrupto (fallo simulado)")
        dato = str(payload)
        return " ".join(dato.strip().lower().split())


class CalcularEstadisticas(ProcesadorBase):
    """Calcula media y desviación de una lista de números."""
    @property
    def id_procesador(self) -> str:
        return "stats"

    async def procesar(self, payload: Any) -> Any:
        await asyncio.sleep(self._simular_trabajo(80) / 1000)
        if random.random() < self.tasa_fallo:
            raise ValueError("cálculo inestable (fallo simulado)")
        nums = [float(x) for x in payload]
        if not nums:
            raise ValueError("lista vacía")
        n = len(nums)
        media = sum(nums) / n
        var = sum((x - media) ** 2 for x in nums) / n
        return {"n": n, "media": round(media, 3), "desviacion": round(var**0.5, 3)}


class TransformarDatos(ProcesadorBase):
    """Aplica mapeo de transformación sobre un dict."""
    @property
    def id_procesador(self) -> str:
        return "transformar"

    async def procesar(self, payload: Any) -> Any:
        await asyncio.sleep(self._simular_trabajo(40) / 1000)
        if random.random() < self.tasa_fallo:
            raise ValueError("transformación falló (fallo simulado)")
        if not isinstance(payload, dict):
            raise TypeError("el payload debe ser un dict")
        return {k.upper(): v * 2 if isinstance(v, (int, float)) else v
                for k, v in payload.items()}


# --- Factory de procesadores (Patrón Factory Method) ---
_REGISTRO: dict[str, type[ProcesadorBase]] = {
    "validar": ValidarDatos,
    "normalizar": NormalizarTexto,
    "stats": CalcularEstadisticas,
    "transformar": TransformarDatos,
}


def crear_procesador(tipo: str, tasa_fallo: float = 0.0) -> Procesador:
    """Devuelve la estrategia adecuada según el tipo del job."""
    if tipo not in _REGISTRO:
        raise ValueError(f"tipo de procesador desconocido: {tipo!r}")
    return _REGISTRO[tipo](tasa_fallo=tasa_fallo)


print("Procesadores + factory listos:", list(_REGISTRO))


**Prueba rápida:** instanciamos el factory y ejecutamos un `ValidarDatos` directo (sin reintentos) para comprobar el contrato.


In [ ]:
async def _demo_procesador():
    proc = crear_procesador("stats")
    r = await proc.procesar([10, 20, 30, 40, 50])
    print("Resultado:", r)

await _demo_procesador()


### Paso 3: Retry Decorator + Trabajador Asíncrono

#### 3.1 Decorador `@retry` (Metaprogramación)

Un decorador que envuelve una función `async` con reintentos y **backoff exponencial**: tras cada fallo espera `_base * 2 ** intento` segundos, con un máximo. Registra cada reintento en el log. Este es el corazón de la resiliencia.


In [ ]:
def retry(
    max_jobs: int = 2,
    base_delay: float = 0.05,
    max_delay: float = 0.5,
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Decorador async: reintenta con backoff exponencial.

    Devuelve el resultado y una lista de errores acumulados en cada intento.
    """
    def decorador(func: Callable[..., Any]) -> Callable[..., Any]:
        async def envoltorio(job: Job, *args: Any, **kw: Any) -> tuple[Any, list[str]]:
            errores: list[str] = []
            espera = base_delay
            for intento in range(1, job.intentos_max + 1):
                try:
                    resultado = await func(job, *args, **kw)
                    return resultado, errores
                except Exception as exc:  # noqa: BLE001
                    errores.append(f"[{intento}] {type(exc).__name__}: {exc}")
                    if intento >= job.intentos_max or intento >= max_jobs:
                        break
                    delay = min(espera * (2 ** (intento - 1)), max_delay)
                    await asyncio.sleep(delay)
                    espera *= 2
            raise RuntimeError(f"job {job.id} agotó intentos: {'; '.join(errores)}")
        return envoltorio
    return decorador

print("Decorador @retry definido")


#### 3.2 Trabajador

El `Trabajador` consume jobs de la cola de entrada, los procesa con la estrategia adecuada envuelta por `@retry`, construye un `JobResult` y lo envía al `Agregador`. Cuando recibe la señal de cierre (centinela `None`), termina limpiamente.


In [ ]:
class Trabajador:
    """Worker asíncrono: consume la cola y procesa jobs con retry."""
    def __init__(
        self,
        nombre: str,
        cola_entrada: "asyncio.Queue[Optional[Job]]",
        agregador: "Agregador",
        tasa_fallo: float = 0.2,
    ) -> None:
        self.nombre = nombre
        self.cola = cola_entrada
        self.agregador = agregador
        self.tasa_fallo = tasa_fallo
        self._log = logging.getLogger(f"motor.trabajador.{nombre}")

    @retry(max_jobs=3)
    async def _procesar_con_estrategia(self, job: Job) -> Any:
        """Procesa el job con la estrategia del factory (con reintentos)."""
        proc = crear_procesador(job.tipo, tasa_fallo=self.tasa_fallo)
        resultado = await proc.procesar(job.payload)
        return {"procesador": proc.id_procesador, "datos": resultado}

    async def _procesar_job(self, job: Job) -> None:
        inicio = time.monotonic()
        num_intentos = 1
        errores: list[str] = []
        try:
            resultado, errores = await self._procesar_con_estrategia(job)
            num_intentos = len(errores) + 1
            res = JobResult(
                job_id=job.id, tipo=job.tipo, exito=True,
                resultado=resultado, duracion_s=time.monotonic() - inicio,
                num_intentos=num_intentos, errores=deque(errores),
            )
            self._log.info("OK  job=%s tipo=%s", job.id, job.tipo)
        except Exception as exc:  # noqa: BLE001
            res = JobResult(
                job_id=job.id, tipo=job.tipo, exito=False,
                resultado=None, duracion_s=time.monotonic() - inicio,
                num_intentos=num_intentos,
                errores=deque([str(exc), *errores]),
            )
            self._log.warning("FAIL job=%s tipo=%s -> %s", job.id, job.tipo, exc)
        # Notifica al agregador (patrón Observer)
        self.agregador.on_resultado(res)

    async def run(self) -> None:
        """Bucle principal: consume la cola hasta recibir un centinela None."""
        self._log.info("Iniciado")
        while True:
            item = await self.cola.get()      # espera trabajo
            try:
                if item is None:              # centinela de cierre
                    self.cola.task_done()
                    break
                await self._procesar_job(item)
            finally:
                self.cola.task_done()
        self._log.info("Finalizado")


### Paso 4: Cola de Trabajos + Productor

#### 4.1 `ColaDeTrabajos` (envoltorio genérico)

Envolvemos `asyncio.Queue[Job]` para centralizar el manejo de la cola (metadatos del tipo, métodos `put_job`/`get_job`) y hacer el contrato más claro.


In [ ]:
class ColaDeTrabajos:
    """Envoltorio de asyncio.Queue[Job] con un nombre y metadatos."""
    def __init__(self, nombre: str, maxsize: int = 0) -> None:
        self.nombre = nombre
        self._q: asyncio.Queue[Optional[Job]] = asyncio.Queue(maxsize=maxsize)

    async def poner(self, job: Optional[Job]) -> None:
        await self._q.put(job)

    async def tomar(self) -> Optional[Job]:
        return await self._q.get()

    def senal_terminado(self) -> None:
        self._q.task_done()

    async def unir(self) -> None:
        await self._q.join()

    @property
    def tamanio(self) -> int:
        return self._q.qsize()


print("ColaDeTrabajos lista")


#### 4.2 Productor

Genera jobs de diversos tipos con **rate limiting** (un mínimo de tiempo entre producciones). Al terminar, envía `None` (centinela) por cada trabajador para indicar el cierre de la cola. Los datos simulados provienen de un pequeño generador de universo.


In [ ]:
def generar_payload(tipo: str, i: int) -> Any:
    """Genera un payload de ejemplo según el tipo de procesador."""
    if tipo == "validar":
        return {"id": i, "monto": random.uniform(10, 1000)}
    if tipo == "normalizar":
        return f"  Ejemplo de TEXTO   para el job {i}  "
    if tipo == "stats":
        return [random.randint(1, 100) for _ in range(8)]
    if tipo == "transformar":
        return {"a": i, "b": i * 2, "texto": f"lote-{i}"}
    return {"tipo": tipo, "i": i}


class Productor:
    """Genera jobs y los coloca en la cola con rate limiting."""
    def __init__(
        self,
        cola: "ColaDeTrabajos",
        num_jobs: int = 10,
        min_intervalo_s: float = 0.01,
        num_trabajadores: int = 3,
        tipos: Optional[list[str]] = None,
    ) -> None:
        self.cola = cola
        self.num_jobs = num_jobs
        self.min_intervalo_s = min_intervalo_s
        self.num_trabajadores = num_trabajadores
        self.tipos = tipos or list(_REGISTRO.keys())
        self._log = logging.getLogger("motor.productor")

    async def run(self) -> None:
        """Produce jobs y al final envía las señales de cierre."""
        self._log.info("Produciendo %d jobs ...", self.num_jobs)
        for i in range(self.num_jobs):
            tipo = random.choice(self.tipos)
            job = Job(id=i, tipo=tipo, payload=generar_payload(tipo, i))
            await self.cola.poner(job)
            # Rate limiting: pausa mínima entre producciones
            if self.min_intervalo_s > 0:
                await asyncio.sleep(self.min_intervalo_s)
        # Centinelas: uno por trabajador para cerrar la cola limpiamente
        for _ in range(self.num_trabajadores):
            await self.cola.poner(None)
        self._log.info("Producción completada (%d jobs)", self.num_jobs)


### Paso 5: Agregador (Sink) + Motor

#### 5.1 Agregador

El `Agregador` actúa como **sink** y **observador**: recibe resultados vía `on_resultado()`, acumula métricas en `Stats` y guarda los detalles de cada resultado.


In [ ]:
class Agregador:
    """Sink: consolida resultados y métricas (patrón Observer)."""
    def __init__(self) -> None:
        self.stats = Stats()
        self.resultados: list[JobResult] = []
        self._log = logging.getLogger("motor.agregador")
        self._inicio_monotonic = time.monotonic()

    def on_resultado(self, res: JobResult) -> None:
        """Notificación de un trabajo completado (Observer)."""
        self.resultados.append(res)
        self.stats.registrar(res)

    def finalizar(self) -> Stats:
        self.stats.registrar_tiempo_total(self._inicio_monotonic)
        return self.stats

    def reporte(self) -> str:
        """Genera un reporte de texto con todos los resultados."""
        lineas = ["================ REPORTE DEL MOTOR ================"]
        lineas.append(f"Total de jobs procesados : {len(self.resultados)}")
        for r in self.resultados:
            estado = "OK " if r.exito else "ERR"
            lineas.append(
                f"  [{estado}] job={r.job_id:>3} tipo={r.tipo:<11} "
                f"int={r.num_intentos} t={r.duracion_s:.3f}s"
            )
        lineas.append("------------------------------------------------")
        lineas.append(f"Métricas: {self.stats.resumen()}")
        lineas.append("================================================")
        return "\n".join(lineas)


#### 5.2 Motor de Procesamiento

El `MotorDeProcesamiento` es el orquestador. Usa un **`asyncio.Semaphore`** para limitar cuántos trabajos se procesan realmente de forma simultánea, `asyncio.gather` para lanzar todos los `Trabajador` a la vez, y `asyncio.timeout` como salvaguarda global. Coordina el pipeline completo: lanza el productor y los workers concurrentemente.


In [ ]:
class MotorDeProcesamiento:
    """Orquestador: coordina productor, colas, workers y sink."""
    def __init__(
        self,
        num_trabajadores: int = 3,
        limite_concurrencia: int = 2,
        num_jobs: int = 12,
        tasa_fallo: float = 0.25,
        min_intervalo_s: float = 0.0,
        timeout_total_s: float = 30.0,
    ) -> None:
        self.num_trabajadores = num_trabajadores
        self.limite_concurrencia = limite_concurrencia
        self.num_jobs = num_jobs
        self.tasa_fallo = tasa_fallo
        self.timeout_total_s = timeout_total_s
        self.cola_entrada = ColaDeTrabajos("entrada")
        self.agregador = Agregador()
        self.sem = asyncio.Semaphore(limite_concurrencia)
        self._log = logging.getLogger("motor.motor")

    async def _trabajador_limitado(self, nombre: str, cola: "ColaDeTrabajos",
                                   ) -> None:
        """Wrapper: aplica el semáforo alrededor del ciclo del trabajador."""
        async with self.sem:
            self._log.info("Adquiere semáforo (concurrencia=%d)",
                           self.sem._value if hasattr(self.sem, "_value") else 0)
            await cola.unir()

    async def ejecutar(self) -> Stats:
        """Ejecuta el pipeline completo y devuelve las métricas finales."""
        self._log.info("Iniciando motor: workers=%d, concurrencia=%d, jobs=%d",
                       self.num_trabajadores, self.limite_concurrencia,
                       self.num_jobs)

        # Cola secundaria / DLQ: por simplicidad compartimos la cola de entrada
        # en este demo, pero el diseño está listo para encadenar una DLQ real.
        productor = Productor(
            self.cola_entrada, num_jobs=self.num_jobs,
            min_intervalo_s=min(0.005, self.tasa_fallo),
            num_trabajadores=self.num_trabajadores,
        )
        trabajadores = [
            Trabajador(f"w{i}", self.cola_entrada, self.agregador,
                       tasa_fallo=self.tasa_fallo)
            for i in range(self.num_trabajadores)
        ]

        inicio = time.monotonic()
        try:
            async with asyncio.timeout(self.timeout_total_s):
                # Lanzamos productor y todos los workers en paralelo
                await asyncio.gather(
                    productor.run(),
                    *(t.run() for t in trabajadores),
                )
        except asyncio.TimeoutError:
            self._log.error("TIEMPO AGOTADO: el motor excedió %ss",
                            self.timeout_total_s)

        self._log.info("Pipeline completado en %.2fs", time.monotonic() - inicio)
        return self.agregador.finalizar()


#### 5.3 Nota sobre la concurrencia con Semaphore

En este diseño el semáforo se aplica a *grupos* de trabajadores, pero el patrón más común en el mundo real es **envolver la sección crítica de cada job** con el semáforo. En el ejercicio 1 verás la variante donde cada `_procesar_job` se ejecuta dentro de `async with self.sem`, limitando los trabajos simultáneos.

Para visualizar la diferencia:

```ascii
  Sin semáforo (todos a la vez)        Con semáforo (máx. 2 simultáneos)
  w1 ────────   w2 ────────            w1 ────►     w2 ────►
  w3 ────────   w4 ────────            w3        w4        (esperan su turno)
```


### Paso 6: Métricas y Reporte Final

Aquí definimos funciones auxiliares para presentar métricas de forma legible y comparar el rendimiento (concurrencia real vs. tiempo de cómputo acumulado).


In [ ]:
def mostrar_reporte(agregador: Agregador) -> None:
    """Imprime el reporte legible del agregador."""
    print(agregador.reporte())


def factor_aceleracion(agregador: Agregador) -> float:
    """Tiempo de cómputo acumulado / tiempo real (speedup por concurrencia)."""
    s = agregador.stats
    if s.tiempo_total_s <= 0:
        return 0.0
    return s.tiempos_totales_s / s.tiempo_total_s


print("Utilidades de reporte listas")


### Paso 7: Orquestación Principal (`main` async)

Función `main()` que instancia el motor, lo ejecuta y muestra el reporte final. La ejecutamos con `await` en el contexto de notebook (equivalente a `asyncio.run(main())`).


In [ ]:
async def main() -> Stats:
    # Parámetros del escenario: 12 jobs, 3 trabajadores, concurrencia real 2
    motor = MotorDeProcesamiento(
        num_trabajadores=3,
        limite_concurrencia=2,
        num_jobs=12,
        tasa_fallo=0.25,
        min_intervalo_s=0.0,
        timeout_total_s=20.0,
    )
    stats = await motor.ejecutar()
    mostrar_reporte(motor.agregador)
    print(f"Factor de aceleración (speedup) : {factor_aceleracion(motor.agregador):.2f}x")
    return stats

# Nota: en un script real usaríamos:  asyncio.run(main())
stats_final = await main()


## 6. Testing del Motor (`unittest.IsolatedAsyncioTestCase`)

Probamos los componentes clave sin depender de paquetes externos usando `IsolatedAsyncioTestCase` de la stdlib: verificamos validez de dominio, el decorador `@retry`, el factory y el pipeline completo del motor.


In [ ]:
import unittest


class TestDominio(unittest.TestCase):
    """Pruebas de las dataclasses (Job, JobResult, Stats)."""

    def test_job_frozen_e_valido(self) -> None:
        j = Job(id=1, tipo="stats", payload=[1, 2, 3])
        self.assertEqual(j.intentos_max, 3)
        # frozen=True impide mutación
        with self.assertRaises(AttributeError):
            j.id = 99  # type: ignore[misc]

    def test_job_intentos_max_invalido(self) -> None:
        with self.assertRaises(ValueError):
            Job(id=1, tipo="x", payload=1, intentos_max=0)

    def test_stats_registra(self) -> None:
        st = Stats()
        st.registrar(JobResult(job_id=1, tipo="a", exito=True))
        st.registrar(JobResult(job_id=2, tipo="a", exito=False, num_intentos=3))
        self.assertEqual(st.completados, 1)
        self.assertEqual(st.fallidos, 1)
        self.assertEqual(st.reintentos, 2)
        self.assertEqual(st.por_tipo, {"a": 2})


class TestFactory(unittest.TestCase):
    """Pruebas del factory de procesadores."""

    def test_crear_procesador_valido(self) -> None:
        proc = crear_procesador("stats")
        self.assertIsInstance(proc, CalcularEstadisticas)
        self.assertEqual(proc.id_procesador, "stats")

    def test_tipo_desconocido(self) -> None:
        with self.assertRaises(ValueError):
            crear_procesador("no_existe")


class TestRetryAsync(unittest.IsolatedAsyncioTestCase):
    """Pruebas del decorador @retry con backoff."""

    async def test_retry_tiene_exito(self) -> None:
        llamadas = 0

        @retry(max_jobs=3, base_delay=0.0)
        async def fr(job: Job) -> str:
            nonlocal llamadas
            llamadas += 1
            if llamadas < 2:
                raise RuntimeError("primer intento falla")
            return "ok"

        job = Job(id=1, tipo="test", payload=None, intentos_max=3)
        res, errores = await fr(job)
        self.assertEqual(res, "ok")
        self.assertEqual(len(errores), 1)   # falló 1 vez

    async def test_retry_agota_intentos(self) -> None:
        @retry(max_jobs=2, base_delay=0.0)
        async def fr(job: Job) -> str:
            raise ValueError("siempre falla")

        job = Job(id=1, tipo="test", payload=None, intentos_max=2)
        with self.assertRaises(RuntimeError):
            await fr(job)


class TestMotorAsync(unittest.IsolatedAsyncioTestCase):
    """Prueba de integración del pipeline completo."""

    async def test_motor_procesa_todos_los_jobs(self) -> None:
        motor = MotorDeProcesamiento(
            num_trabajadores=3,
            limite_concurrencia=2,
            num_jobs=10,
            tasa_fallo=0.0,      # sin fallos: todo debe completar
            min_intervalo_s=0.0,
            timeout_total_s=10.0,
        )
        stats = await motor.ejecutar()
        self.assertEqual(stats.completados, 10)
        self.assertEqual(stats.fallidos, 0)

    async def test_motor_con_fallos_usa_dlq_concepto(self) -> None:
        motor = MotorDeProcesamiento(
            num_trabajadores=2,
            limite_concurrencia=1,
            num_jobs=6,
            tasa_fallo=1.0,      # 100% fallo: todos deben quedar en error
            min_intervalo_s=0.0,
            timeout_total_s=10.0,
        )
        stats = await motor.ejecutar()
        self.assertGreaterEqual(stats.fallidos, 1)
        self.assertEqual(stats.completados, 0)


**Ejecutamos la suite completa de pruebas:**


In [ ]:
resultado = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromModule(
        __import__(__name__) if __name__ != "__main__" else __import__("__main__")
    )
)
print("\n¿OK todo?", resultado.wasSuccessful())


## 7. Ejecución Completa del Motor (Escenario Realista)

Ahora ejecutamos el motor con más jobs, una tasa de fallo moderada y tiempo minúsculo de intervalos, para observar el comportamiento real: reintentos, DLQ (fallos) y métricas.


In [ ]:
async def escenario_completo() -> Stats:
    motor = MotorDeProcesamiento(
        num_trabajadores=4,
        limite_concurrencia=3,
        num_jobs=20,
        tasa_fallo=0.30,
        min_intervalo_s=0.005,
        timeout_total_s=30.0,
    )
    stats = await motor.ejecutar()
    mostrar_reporte(motor.agregador)
    print(f"Speedup (concurrencia real)    : {factor_aceleracion(motor.agregador):.2f}x")
    return stats

stats_escenario = await escenario_completo()


### Interpretación de las métricas

- **Completados / Fallidos**: qué fracción de trabajos terminó bien. Los fallidos agotaron sus reintentos y quedan "apartados" (conceptualmente en la DLQ).
- **Reintentos**: cuántas veces se reintentó gracias a `@retry`. Un valor > 0 confirma que el backoff funcionó.
- **Speedup**: cociente entre el tiempo de cómputo *acumulado* (si fuera serie) y el tiempo *real* en pared. Un valor > 1 demuestra el beneficio de la concurrencia.


## 8. Extensiones Posibles

El motor está diseñado para crecer en varias dimensiones:

| Extensión | Cómo |
|-----------|------|
| **Persistencia en Base de Datos** | El `Agregador` (sink) puede escribir `JobResult` a SQLite/Postgres en lugar de solo imprimir. |
| **Parallelismo REAL (CPU)** | Combinar `asyncio` (I/O) con `concurrent.futures.ProcessPoolExecutor` para tareas intensivas en CPU: envolver en `loop.run_in_executor()`. |
| **Dead Letter Queue real** | Encadenar una segunda cola (`ColaDeTrabajos("dlq")`) que recoja los jobs fallidos para reprocesarlos o inspeccionarlos. |
| **Monitores / Dashboard** | Un `Monitor` (Observer adicional) que muestre en vivo el progreso de la cola y las métricas parciales. |
| **Prioridades** | `asyncio.PriorityQueue` y ordenar por `job.prioridad`. |
| **Cancelación / shutdown** | `asyncio.TaskGroup` (3.11+) con `cancel` para detener graceful el pipeline. |
| **Distribuido** | Sustituir `Queue` local por Redis/AMQP para workers en múltiples nodos. |
| **Backpressure** | Ya presente con `maxsize` en la cola: el productor bloquea al llenarse la cola.


## 9. Ejercicios Extra (2 Retos)

### Ejercicio 1: Semáforo por Job (concurrencia fina)

Modifica el `MotorDeProcesamiento` para que el **semáforo limite cada job individual**, no grupos de workers. Concretamente:

1. Añade al `Trabajador._procesar_job` la ejecución de la sección crítica dentro de `async with sem:`.
2. Ajusta el semáforo para que el worker adquiera el bloqueo solo durante el proceso de un job.
3. Con `num_trabajadores=4` y `limite_concurrencia=2`, ejecuta `num_jobs=20`. Comprueba que **nunca hay más de 2 jobs procesándose a la vez** (puedes imprimir el valor del semáforo o contar máximos simultáneos con un contador).

> Pista: usa un contador global con `lock` o una variable de clase con `max()` para llevar el pico de concurrencia.

### Ejercicio 2: Procesador con Parallelismo CPU (ProcessPoolExecutor)

Crea un nuevo procesador `CalcularPesado` que simule trabajo intensivo en CPU (p. ej. `sum(i*i for i in range(N))`) y ejecútalo a través de un `ProcessPoolExecutor` combinado con `asyncio`:

1. Registra el nuevo tipo en el `_REGISTRO` del factory.
2. Dentro de `procesar`, usa `asyncio.to_thread` o `loop.run_in_executor` con un `ProcessPoolExecutor` para lanzar la tarea pesada sin bloquear el bucle de eventos.
3. Genera jobs de tipo `pesado` y compara el speedup antes/después del cambio.

> Pista: `await asyncio.to_thread(fn_pesada, payload)` es la forma más simple en 3.9+. Para CPU real usa `ProcessPoolExecutor` con `loop.run_in_executor`.


## 10. Resumen y Lecciones Aprendidas

Construimos un **Motor de Procesamiento Asíncrono de Datos** completo, integrando todas las habilidades del nivel Avanzado:

| Habilidad del curso | Dónde la aplicamos |
|---------------------|--------------------|
| **Asyncio avanzado** | `Semaphore`, `Queue`, `gather`, `timeout`, `asyncio.run`, corrutinas, `sleep` en `Productor`/`Trabajador`/`Procesador` |
| **Tipado (typing)** | `Protocol`, `Optional`, `Callable`, `Any`, genéricos `Queue[Optional[Job]]`, anotaciones en todas las firmas |
| **POO avanzada** | Herencia (`ProcesadorBase`), abstracción (`ABC`, `abstractmethod`), composición (`Trabajador` → `Procesador` → `Agregador`), polimorfismo |
| **Dataclasses** | `Job` (frozen, `__post_init__`), `JobResult`, `Stats` (`field`, `default_factory`) |
| **Logging** | `logging` con formato estructurado y loggers por componente |
| **Metaprogramación** | Decorador `@retry` (función de orden superior), decoradores propios |
| **Patrones de diseño** | Strategy (`Procesador`), Factory (`crear_procesador`), Observer (`Agregador.on_resultado`), Resiliencia con backoff |
| **Concurrent.futures** | Propuesto en Ejercicio 2 con `ProcessPoolExecutor` para CPU-bound |
| **Excepciones** | Manejo resiliente, `try/except`, `raise`, excepciones personalizadas en dominio |
| **Testing** | `unittest.IsolatedAsyncioTestCase`, tests unitarios e integración |
| **Time / métricas** | `time.monotonic`, contadores, reportes, speedup |

### Lecciones clave

1. **Separar responsabilidades**: productor, cola, workers y sink son piezas independientes y reutilizables.
2. **La concurrencia asíncrona brilla con I/O**: `asyncio` es ideal cuando hay espera; el speedup lo demuestra.
3. **Los semáforos evitan saturar recursos** (BD, APIs, disco) limitando simultaneidad.
4. **La resiliencia se diseña, no se improvisa**: retry + backoff + DLQ forman un patrón robusto.
5. **El tipado + Protocol + POO** hacen el sistema extensible: añadir un procesador es solo registrar una clase.

### ¿Qué sigue?

Con este motor puedes extenderlo para preprocesar datos reales, construir un ETL asíncrono, o integrarlo con bases de datos y APIs. ¡El nivel Avanzado está completado! 🚀
